# Predicciones LightGBM Multi-Horizonte

Genera predicciones con LightGBM (18 horizontes) y crea la tabla `fact_predicciones` en PostgreSQL.

## Dimensiones reutilizadas del datamart
- `dim_tiempo`, `dim_riesgo`, `dim_sector`, `dim_sucursal`

## Entradas
- `output/datasets/datos_preprocesados.csv`
- Modelos: `output/modelos_lightgbm/modelo_lgbm_h*.txt`

## Salidas
- `fact_predicciones` en PostgreSQL
- `output/predicciones/predicciones.csv`

In [43]:
%load_ext autoreload
%autoreload 2

import json
import logging
import os
import sys
from datetime import datetime

import lightgbm as lgb
import numpy as np
import pandas as pd
import psycopg2

sys.path.insert(0, '..')
from src.sql import (
    SQL_CREA_FACT_PREDICCIONES,
    SQL_INSERT_PREDICCIONES,
    SQL_REFRESH_DIM_RIESGO,
    SQL_REFRESH_DIM_SECTOR,
    SQL_REFRESH_DIM_SUCURSAL,
    SQL_REFRESH_DIM_TIEMPO,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

## Configuración

In [44]:
PATH_INPUT = "output/datasets/datos_preprocesados.csv"
PATH_LGBM = "output/modelos_lightgbm"
PATH_OUTPUT = "output/predicciones"
os.makedirs(PATH_OUTPUT, exist_ok=True)

DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "postgres_db",
    "user": "postgres_usr",
    "password": "admin123",
}

MAX_HORIZONTE = 18
VENTANA_LGBM = 6

print(f"Horizontes: {MAX_HORIZONTE}")
print(f"Ventana LGBM: {VENTANA_LGBM} meses")

Horizontes: 18
Ventana LGBM: 6 meses


## Carga de datos y modelos

In [45]:
df = pd.read_csv(PATH_INPUT)
df['mes'] = pd.to_datetime(df['mes'])

with open(os.path.join(PATH_LGBM, 'config_lgbm_18m.json')) as f:
    config_lgbm = json.load(f)

features_numericas = config_lgbm['features_numericas']

modelos_lgbm = []
for h in range(1, MAX_HORIZONTE + 1):
    modelos_lgbm.append(
        lgb.Booster(model_file=os.path.join(PATH_LGBM, f'modelo_lgbm_h{h}.txt'))
    )

print(f"Datos: {len(df):,} registros, {df['bloque_id'].nunique()} bloques")
print(f"Modelos LightGBM: {len(modelos_lgbm)}")

Datos: 20,025 registros, 950 bloques
Modelos LightGBM: 18


## Funciones

In [46]:
def crear_secuencias_lgbm(df, bloque_id, features, ventana):
    """Genera secuencias estadísticas para LightGBM."""
    df_b = df[df['bloque_id'] == bloque_id].sort_values('mes')
    if len(df_b) < ventana:
        return None, None, None

    X_seq, meses = [], []
    for i in range(len(df_b) - ventana + 1):
        hist = df_b[features].iloc[i:i+ventana]
        feats = []
        for col in features:
            v = hist[col].values
            feats.extend([
                np.mean(v), np.std(v), np.min(v), np.max(v),
                np.median(v), v[-1], v[-1] - v[0],
            ])
        X_seq.append(feats)
        meses.append(df_b['mes'].iloc[i + ventana - 1])
    return np.array(X_seq), meses, df_b


print("Funciones definidas.")

Funciones definidas.


In [47]:
# Actualizar todas las dimensiones desde el CSV
conn_tmp = psycopg2.connect(
    host=DB_CONFIG['host'], port=DB_CONFIG['port'],
    dbname=DB_CONFIG['database'],
    user=DB_CONFIG['user'], password=DB_CONFIG['password'],
)
conn_tmp.autocommit = True
cur = conn_tmp.cursor()

fechas_csv = df['mes'].sort_values().unique()
for fecha in fechas_csv:
    cur.execute(SQL_REFRESH_DIM_TIEMPO, (
        fecha, fecha.year, (fecha.month - 1) // 3 + 1,
        fecha.month, fecha.strftime('%B'),
    ))

for riesgo in df['riesgo'].unique():
    cur.execute(SQL_REFRESH_DIM_RIESGO, (str(riesgo), str(riesgo)))

for sector in df['sector'].unique():
    cur.execute(SQL_REFRESH_DIM_SECTOR, (str(sector), str(sector)))

for suc in df['codigo_sucursal'].unique():
    cur.execute(SQL_REFRESH_DIM_SUCURSAL, (int(suc),))

cur.close()
conn_tmp.close()
print("Dimensiones actualizadas desde CSV.")
print(f"  Fechas: {len(fechas_csv)}, Riesgos: {df['riesgo'].nunique()}, "
      f"Sectores: {df['sector'].nunique()}, Sucursales: {df['codigo_sucursal'].nunique()}")

Dimensiones actualizadas desde CSV.
  Fechas: 48, Riesgos: 1, Sectores: 20, Sucursales: 64


## Generar predicciones

In [48]:
predicciones = []

for bloque in df['bloque_id'].unique():
    info = df[df['bloque_id'] == bloque]
    riesgo = info['riesgo'].iloc[0]
    sector = info['sector'].iloc[0]
    sucursal = int(info['codigo_sucursal'].iloc[0])

    X_seq, meses_seq, _ = crear_secuencias_lgbm(
        df, bloque, features_numericas, VENTANA_LGBM
    )
    if X_seq is None or len(X_seq) == 0:
        continue

    for idx in range(len(X_seq)):
        row = {
            'bloque_id': bloque, 'riesgo': riesgo, 'sector': sector,
            'codigo_sucursal': sucursal, 'mes_prediccion': meses_seq[idx],
        }
        x_in = X_seq[idx:idx+1]
        for h in range(MAX_HORIZONTE):
            p = float(modelos_lgbm[h].predict(x_in)[0])
            row[f'prob_h{h+1}'] = round(p, 6)
            row[f'pred_h{h+1}'] = int(p > 0.5)
        predicciones.append(row)

df_pred = pd.DataFrame(predicciones)

prob_cols = [f'prob_h{i+1}' for i in range(MAX_HORIZONTE)]
pred_cols = [f'pred_h{i+1}' for i in range(MAX_HORIZONTE)]

df_pred['prob_media'] = df_pred[prob_cols].mean(axis=1).round(6)
df_pred['pred_media'] = (df_pred['prob_media'] > 0.5).astype(int)
df_pred['crisis_count'] = df_pred[pred_cols].sum(axis=1).astype(int)
df_pred['fecha_ejecucion'] = datetime.now()

print(f"Predicciones: {len(df_pred):,}")
print(f"\nDistribucion pred_media:")
print(df_pred['pred_media'].value_counts().to_string())

Predicciones: 15,729

Distribucion pred_media:
pred_media
0    12690
1     3039


In [49]:
csv_path = os.path.join(PATH_OUTPUT, 'predicciones.csv')
df_pred.to_csv(csv_path, index=False)
print(f"CSV guardado: {csv_path}")

CSV guardado: output/predicciones/predicciones.csv


## Crear tabla `fact_predicciones`

In [50]:
conn = psycopg2.connect(
    host=DB_CONFIG['host'], port=DB_CONFIG['port'],
    dbname=DB_CONFIG['database'],
    user=DB_CONFIG['user'], password=DB_CONFIG['password'],
)
conn.autocommit = True

cur = conn.cursor()
cur.execute(SQL_CREA_FACT_PREDICCIONES)
print("Tabla fact_predicciones creada.")
cur.close()

Tabla fact_predicciones creada.


## Poblar tabla

In [51]:
cur = conn.cursor()
inserted = 0

for _, row in df_pred.iterrows():
    riesgo_s = str(row['riesgo'])
    sector_s = str(row['sector'])
    sucursal_i = int(row['codigo_sucursal'])

    params = (
        row['mes_prediccion'], riesgo_s, sector_s, sucursal_i,
        row['bloque_id'],
    )
    for h in range(1, MAX_HORIZONTE + 1):
        params += (row[f'prob_h{h}'],)
    for h in range(1, MAX_HORIZONTE + 1):
        params += (int(row[f'pred_h{h}']),)
    params += (row['prob_media'], int(row['pred_media']),
              int(row['crisis_count']), row['fecha_ejecucion'])

    cur.execute(SQL_INSERT_PREDICCIONES, params)
    inserted += cur.rowcount

conn.commit()
cur.close()
print(f"Filas insertadas/actualizadas: {inserted:,}")

Filas insertadas/actualizadas: 15,729


## Validación

In [52]:
print("=" * 60)
print("VALIDACION DE FACT_PREDICCIONES")
print("=" * 60)

cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM fact_predicciones")
print(f"\nTotal registros: {cur.fetchone()[0]:,}")

cur.execute("SELECT pred_media, COUNT(*) FROM fact_predicciones GROUP BY pred_media ORDER BY pred_media")
print("\nDistribucion pred_media:")
for r in cur.fetchall():
    label = "Sin crisis" if r[0] == 0 else "Crisis"
    print(f"  {label}: {r[1]:,}")

cur.execute("""
    SELECT dr.descripcion, COUNT(*) 
    FROM fact_predicciones fp JOIN dim_riesgo dr ON fp.id_riesgo = dr.id_riesgo 
    GROUP BY dr.descripcion ORDER BY dr.descripcion
""")
print("\nPor riesgo:")
for r in cur.fetchall():
    print(f"  {r[0]}: {r[1]:,}")

cur.execute("""
    SELECT dt.mes, 
    ROUND(AVG(fp.prob_media)::numeric, 4), 
    SUM(fp.pred_media) 
    FROM fact_predicciones fp 
    JOIN dim_tiempo dt ON fp.id_tiempo = dt.id_tiempo 
    GROUP BY dt.mes ORDER BY dt.mes LIMIT 10
""")
print("\nMuestra por mes:")
print(f"{'mes':<12} {'prob_media':>12} {'crisis_pred':>12}")
print("-" * 40)
for r in cur.fetchall():
    print(f"{str(r[0]):<12} {r[1]:>12} {r[2]:>12}")

cur.close()
print("\n" + "=" * 60)

VALIDACION DE FACT_PREDICCIONES

Total registros: 15,729

Distribucion pred_media:
  Sin crisis: 12,690
  Crisis: 3,039

Por riesgo:
  2: 15,729

Muestra por mes:
mes            prob_media  crisis_pred
----------------------------------------
2019-06-01         0.2324           42
2019-07-01         0.2313           49
2019-08-01         0.2251           47
2019-09-01         0.2265           49
2019-10-01         0.2488           63
2019-11-01         0.2692           53
2019-12-01         0.1395           11
2020-01-01         0.0792            0
2020-02-01         0.0821            1
2020-03-01         0.0849            1



In [53]:
cur = conn.cursor()
cur.execute("""
    SELECT bloque_id,
           ROUND(prob_media::numeric, 4) AS prob,
           pred_media,
           crisis_count
    FROM fact_predicciones
    ORDER BY prob_media DESC
    LIMIT 10
""")
print("Top 10 mayor probabilidad de crisis:")
print(f"{'bloque_id':<15} {'prob':>8} {'pred':>6} {'#crisis':>8}")
print("-" * 40)
for r in cur.fetchall():
    print(f"{r[0]:<15} {r[1]:>8} {r[2]:>6} {r[3]:>8}")
cur.close()

Top 10 mayor probabilidad de crisis:
bloque_id           prob   pred  #crisis
----------------------------------------
2_44_26           0.8189      1       18
2_44_8            0.8177      1       17
2_44_15           0.8170      1       18
2_48_7            0.8167      1       18
2_44_26           0.8167      1       18
2_48_8            0.8162      1       18
2_44_15           0.8162      1       18
2_48_7            0.8160      1       18
2_44_1            0.8157      1       17
2_44_49           0.8154      1       17


In [54]:
conn.close()
print("Conexion cerrada.")

Conexion cerrada.


![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*